In [35]:
import sys
import os
import json
import faiss
import pymupdf
import numpy as np

from tqdm import tqdm
from sentence_transformers import SentenceTransformer

import ollama

embedder = SentenceTransformer(
    "paraphrase-multilingual-mpnet-base-v2"
)

ollama_model = "gemma3:4b"

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [37]:
# Tworzymy pusty indeks FAISS typu FlatL2 – przechowuje wektory i szuka po odległości euklidesowej
# get_embedding_dimension() zwraca wymiar wektorów modelu embeddingowego (768)
index = faiss.IndexFlatL2(embedder.get_embedding_dimension())

# Lista słowników przechowująca metadane każdego chunka (nazwa pliku, numer strony, tekst)
metadata = []

print('Number of chunks: ', index.ntotal) # 0

Number of chunks:  0


In [39]:
class Utils:
    def __init__(self, embedding_model=None, ollama_model="phi3:mini", index=None, metadata=None, chunk_size=512):
        self.embedding_model = embedding_model
        self.ollama_model = ollama_model
        self.index = index
        self.metadata = metadata
        self.chunk_size = chunk_size

    def extract_text_from_pdf(self, pdf_path):
        """
        Extract text from PDF file. Returns a list of tuples (page_number, text).
        """
        text = []
        pdf_document = pymupdf.open(pdf_path)
        for page_num in range(len(pdf_document)):
            page = pdf_document.load_page(page_num)
            # Zamieniamy znaki nowej linii na spacje, żeby tekst był ciągły
            text.append((page_num, str(page.get_text()).replace("\n", " ")))
        return text

    def chunk_text(self, text: list[tuple[int, str]]):
        chunks = []
        for page_num, page_text in text:
            # Dzielimy tekst strony na fragmenty o długości chunk_size znaków
            page_chunks = [
                (page_num, page_text[i:i+self.chunk_size])
                for i in range(0, len(page_text), self.chunk_size)
            ]
            chunks.extend(page_chunks)
        return chunks

    def add_chunks_to_faiss(self, chunks, filename, db_loc="vec_db/"):
        for chunk_num, (page_number, chunk) in enumerate(tqdm(chunks, desc="Adding chunks to FAISS")):
            # Zamieniamy tekst chunka na wektor embeddingowy
            embeddings = self.embedding_model.encode(chunk, show_progress_bar=False)
            # Dodajemy wektor do indeksu FAISS
            self.index.add(np.array([embeddings]))
            # Zapisujemy metadane chunka – powiążemy je z wektorem przez pozycję w indeksie
            self.metadata.append({
                "filename": filename,
                "page_number": page_number,
                "chunk_num": chunk_num,
                "chunk": chunk
            })
        # Zapisujemy indeks FAISS na dysk, żeby nie trzeba było go odbudowywać przy każdym uruchomieniu
        faiss.write_index(self.index, db_loc + "vector_database.index")
        with open(db_loc + "metadata.json", "w") as file:
            json.dump(self.metadata, file)

    def process_file(self, file_path):
        """
        Process the file and add chunks to FAISS index
        """
        if file_path.endswith('.pdf'):
            text = self.extract_text_from_pdf(file_path)
        else:
            print(f"Unsupported file format, with extension: {os.path.splitext(file_path)[1]}")
            return 0

        chunks = self.chunk_text(text)
        self.add_chunks_to_faiss(chunks, filename=os.path.basename(file_path))
        return len(chunks)
    
    def answer_question(self, prompt_template="", query="", max_tokens=512, temp=0.7, k=5):
        # Zamieniamy pytanie użytkownika na wektor embeddingowy
        question_embedding = self.embedding_model.encode(query, show_progress_bar=False)

        # Szukamy k najbliższych wektorów w FAISS – D to odległości, I to indeksy znalezionych chunków
        D, I = self.index.search(np.array([question_embedding]), k)
        # Pobieramy metadane (tekst) znalezionych chunków
        chunks = [self.metadata[i] for i in I[0]]

        # Sklejamy teksty chunków w jeden blok kontekstu dla modelu
        context = ""
        for i, chunk in enumerate(chunks):
            context += f"{i+1}. {chunk['chunk']}\n"

        # Wstawiamy kontekst i pytanie do szablonu promptu
        prompt = prompt_template.format(context=context, query=query)

        messages = [
            {
                "role": "system",
                "content": (
                    "Be helpful, straight to the point. "
                    "Use only context. Do not hallucinate."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
        
        response = ollama.chat(
            model=self.ollama_model,
            messages=messages,
            options={
                "temperature": temp,
                "num_predict": max_tokens
            }
        )
        
        answer = response["message"]["content"]

        return answer, chunks

In [41]:
# Tworzymy obiekt Utils łącząc wszystkie komponenty RAG w jednym miejscu
utils = Utils(
    embedding_model=embedder,
    ollama_model="gemma3:4b",
    index=index,
    metadata=metadata
)

In [43]:
knowledge_dir = "knowledge/"
# Przetwarzamy każdy plik PDF z katalogu – dzielimy na chunki i dodajemy do FAISS
for file in os.listdir(knowledge_dir):
    utils.process_file(knowledge_dir + file)
    
print('Number of chunks: ', index.ntotal)

Adding chunks to FAISS: 100%|██████████| 20/20 [00:00<00:00, 26.15it/s]

Number of chunks:  259


In [63]:
# przykładowe pytania dotyczace powyzszych dokumentow
questions = [
    "Który obszar systemów CAx odpowiada za przeprowadzanie analiz inżynierskich, takich jak Metoda Elementów Skończonych (MES)?",
    "Która cecha najlepiej opisuje systemy 3D, takie jak CATIA V5?",
    "Dlaczego należy unikać odniesień typu B-Rep?",
    "Zgodnie z zasadami modelowania, jak powinno wykonywać się zaokrąglenia?",
    "Która operacja wymaga profilu oraz krzywej prowadzącej (Center Curve)?",
    "Gdzie standardowo tworzy się elementy modelowania powierzchniowego?",
    "Co oznacza ciągłość C2?",
    "Jaka jest najważniejsza różnica między krzywymi Béziera i B-spline?",
    "Jaki jest główny cel stosowania publikacji w złożeniach?",
    "Który więz powinien zostać zastosowany do pierwszego elementu złożenia?",
    "Co umożliwia funkcja Scene w CATIA V5?",
    "Który element ma największy wpływ na realizm renderingu?",
    "Który widok stanowi podstawę do tworzenia pozostałych rzutów w module DRAFTING?",
    "Co oznacza skrót BOM?",
    "Co oznacza asocjatywność wymiarów w modelu 3D?",
    "Dlaczego wykrywanie małych egzoplanet jest trudniejsze niż dużych?",
]

In [47]:
from IPython.display import display, Markdown

prompt_template ="""Based on the following context items, please answer the query.
Give yourself room to think by extracting relevant passages from the context before answering the query.
Don't return the thinking, only return the answer.
You must answer using only information explicitly present in the context.
If the answer is not directly stated or cannot be inferred with high confidence from the context, respond exactly:
"Nie znaleziono informacji w dostarczonych dokumentach."
Do not use prior knowledge.
Do not guess.
Do not supplement missing information.
Do not explain why information is missing.
Answer in Polish language only.
Use the following examples as reference for the ideal answer style.
Example 1:
Pytanie: Dlaczego Księżyc zawsze pokazuje tę samą stronę Ziemi?
Księżyc pokazuje Ziemi zawsze tę samą stronę, ponieważ jest związany pływowo z Ziemią. Oznacza to, że jego czas obrotu wokół własnej osi jest równy czasowi obiegu wokół Ziemi (około 27,3 dnia). W wyniku działania sił grawitacyjnych Ziemi rotacja Księżyca została w przeszłości spowolniona aż do osiągnięcia tego stanu równowagi.
Now use the following context items to answer this one user query only:
{context}
Relevant passages: <extract relevant passages from the context here>
Main User Query: {query}
Answer:\n"""

# Wybieramy losowo zapytanie z listy
random_query = np.random.choice(questions)

response, chunks = utils.answer_question(
        prompt_template=prompt_template,
        query=random_query,
        max_tokens=512,
        temp=0.1
)

display(Markdown(f"**Pytanie:** {random_query}"))
display(Markdown(f"**Odpowiedź:**\n\n{response}"))
display(Markdown("---\n**Źródła:**"))
for i, chunk in enumerate(chunks):
    excerpt = chunk['chunk'][:200].strip() + "..."
    display(Markdown(
        f"**[{i+1}]** `{chunk['filename']}` — strona {chunk['page_number'] + 1}\n\n"
        f"> {excerpt}"
    ))

**Pytanie:** Co oznacza ciągłość C2?

**Odpowiedź:**

Ciągłość 𝒞2, czyli ciągłość samej krzywej (brak przerw) wynikająca z warunku łączenia się segmentów, ciągłość pierwszej pochodnej, czyli warunek styczności decydujący o gładkości krzywej.


---
**Źródła:**

**[1]** `Wyklad 5.pdf` — strona 25

> wa limity (ograniczenia) • Granica i środek • Z powierzchnią odniesienia • Z powierzchnią styczności • Z krzywą odniesienia • Z dwiema powierzchniami styczności • Z kierunkiem pochylenia...

**[2]** `Wyklad 6.pdf` — strona 13

> raku ostrzy) i ciągłość drugiej pochodnej (ciągłość krzywizny), która  nie daje efektu wizualnego jak poprzednie ciągłości. Schemat składania krzywej B-sklejanej z krzywych Béziera i związki między pu...

**[3]** `Wyklad 6.pdf` — strona 13

> Interpolacja krzywych w systemach CAD Krzywe B-spline Krzywa B-spline 𝒔(𝑡) stopnia 𝑛 powstaje z połączenia 𝑀 krzywych Béziera tego samego  stopnia 𝑛, tak by w punktach łączenia miała miejsce równość p...

**[4]** `Wyklad 6.pdf` — strona 9

> niczony do odcinka (prostej), okręgu (łuku), elipsy (łuku eliptycznego) oraz krzywej  B-splajn zwanej popularnie splajnem. Odcinek Łuk Elipsa „Splajn” Catia V5 ma do dyspozycji kilka opcji więcej niż...

**[5]** `Wyklad 6.pdf` — strona 14

> są punktami przecięcia przedłużonych  odcinków łamanej kontrolnej Béziera z segmentu poprzedzającego 𝑭𝑖−2𝑮𝑖−2  i z segmentu  następującego 𝑭𝑖−1𝑮𝑖−1 – taki dobór jest jednoznaczny. INŻYNIERSKIE PAKIETY...

In [49]:
from IPython.display import display, Markdown

prompt_template ="""Based on the following context items, please answer the query.
Give yourself room to think by extracting relevant passages from the context before answering the query.
Don't return the thinking, only return the answer.
You must answer using only information explicitly present in the context.
If the answer is not directly stated or cannot be inferred with high confidence from the context, respond exactly:
"Nie znaleziono informacji w dostarczonych dokumentach."
Do not use prior knowledge.
Do not guess.
Do not supplement missing information.
Do not explain why information is missing.
Answer in Polish language only.
Use the following examples as reference for the ideal answer style.
Example 1:
Pytanie: Dlaczego Księżyc zawsze pokazuje tę samą stronę Ziemi?
Księżyc pokazuje Ziemi zawsze tę samą stronę, ponieważ jest związany pływowo z Ziemią. Oznacza to, że jego czas obrotu wokół własnej osi jest równy czasowi obiegu wokół Ziemi (około 27,3 dnia). W wyniku działania sił grawitacyjnych Ziemi rotacja Księżyca została w przeszłości spowolniona aż do osiągnięcia tego stanu równowagi.
Now use the following context items to answer this one user query only:
{context}
Relevant passages: <extract relevant passages from the context here>
Main User Query: {query}
Answer:\n"""

# Wybieramy losowo zapytanie z listy
random_query = np.random.choice(questions)

response, chunks = utils.answer_question(
        prompt_template=prompt_template,
        query=random_query,
        max_tokens=512,
        temp=0.1
)

display(Markdown(f"**Pytanie:** {random_query}"))
display(Markdown(f"**Odpowiedź:**\n\n{response}"))
display(Markdown("---\n**Źródła:**"))
for i, chunk in enumerate(chunks):
    excerpt = chunk['chunk'][:200].strip() + "..."
    display(Markdown(
        f"**[{i+1}]** `{chunk['filename']}` — strona {chunk['page_number'] + 1}\n\n"
        f"> {excerpt}"
    ))

**Pytanie:** Jaki jest główny cel stosowania publikacji w złożeniach?

**Odpowiedź:**

Wiązania w złożeniach muszą opierać się na opublikowanych źródłach, aby umożliwić podmianę geometrii bez reżeneralnej aktualizacji powiązań.


---
**Źródła:**

**[1]** `Wyklad 7.pdf` — strona 8

> Tworzenie produktów / złożeń Pozycjonowanie – referencje Z publikacjami Bez publikacjami Dzięki zastosowaniu publikacji zyskujemy: 1) Wymienność - pozycjonowanie i linki na podstawie nazw publikacji 2...

**[2]** `Wyklad 2.pdf` — strona 30

> zygotowywany w szkicowniku powinien być możliwie najprostszy. Nie  oznacza to jednak, że powinien składać się jedynie z prostokątów i okręgów. Należy znaleźć  kompromis pomiędzy stopniem szczegółowośc...

**[3]** `Wyklad 1.pdf` — strona 14

> y rozwiazan  parametrycznych (opartych o zdefiniowane zalżności wybranych relacji wymiarowych i  geometrycznych). Podobnie jak przy wczesniejszych programach możliwe jest korzystanie z  bogatych bibli...

**[4]** `Wyklad 6.pdf` — strona 9

> niczony do odcinka (prostej), okręgu (łuku), elipsy (łuku eliptycznego) oraz krzywej  B-splajn zwanej popularnie splajnem. Odcinek Łuk Elipsa „Splajn” Catia V5 ma do dyspozycji kilka opcji więcej niż...

**[5]** `Wyklad 7.pdf` — strona 9

> Tworzenie produktów / złożeń Pozycjonowanie – referencje Więzy oparte na publikacjach pozwalają na podmiane geometrii bez rećznej aktualizacji powiązań. Jedynum  warunkiem jest, żeby publikacje miały...

In [51]:
from IPython.display import display, Markdown

prompt_template ="""Based on the following context items, please answer the query.
Give yourself room to think by extracting relevant passages from the context before answering the query.
Don't return the thinking, only return the answer.
You must answer using only information explicitly present in the context.
If the answer is not directly stated or cannot be inferred with high confidence from the context, respond exactly:
"Nie znaleziono informacji w dostarczonych dokumentach."
Do not use prior knowledge.
Do not guess.
Do not supplement missing information.
Do not explain why information is missing.
Answer in Polish language only.
Use the following examples as reference for the ideal answer style.
Example 1:
Pytanie: Dlaczego Księżyc zawsze pokazuje tę samą stronę Ziemi?
Księżyc pokazuje Ziemi zawsze tę samą stronę, ponieważ jest związany pływowo z Ziemią. Oznacza to, że jego czas obrotu wokół własnej osi jest równy czasowi obiegu wokół Ziemi (około 27,3 dnia). W wyniku działania sił grawitacyjnych Ziemi rotacja Księżyca została w przeszłości spowolniona aż do osiągnięcia tego stanu równowagi.
Now use the following context items to answer this one user query only:
{context}
Relevant passages: <extract relevant passages from the context here>
Main User Query: {query}
Answer:\n"""

# Wybieramy losowo zapytanie z listy
random_query = np.random.choice(questions)

response, chunks = utils.answer_question(
        prompt_template=prompt_template,
        query=random_query,
        max_tokens=512,
        temp=0.1
)

display(Markdown(f"**Pytanie:** {random_query}"))
display(Markdown(f"**Odpowiedź:**\n\n{response}"))
display(Markdown("---\n**Źródła:**"))
for i, chunk in enumerate(chunks):
    excerpt = chunk['chunk'][:200].strip() + "..."
    display(Markdown(
        f"**[{i+1}]** `{chunk['filename']}` — strona {chunk['page_number'] + 1}\n\n"
        f"> {excerpt}"
    ))

**Pytanie:** Co oznacza skrót BOM?

**Odpowiedź:**

BOM to skrót od Bill of Material, czyli spis części.


---
**Źródła:**

**[1]** `Wyklad 2.pdf` — strona 11

> Informacje wstępne B-Rep (Boundary Representation) – powstaje kiedy bezpośrednio w przestrzeni 3D  wybrany zostanie jeden z elementów typu: powierzchnia, krawędź lub punkt. Jest  wynikiem operacji rek...

**[2]** `Wyklad 6.pdf` — strona 13

> raku ostrzy) i ciągłość drugiej pochodnej (ciągłość krzywizny), która  nie daje efektu wizualnego jak poprzednie ciągłości. Schemat składania krzywej B-sklejanej z krzywych Béziera i związki między pu...

**[3]** `Wyklad 6.pdf` — strona 9

> niczony do odcinka (prostej), okręgu (łuku), elipsy (łuku eliptycznego) oraz krzywej  B-splajn zwanej popularnie splajnem. Odcinek Łuk Elipsa „Splajn” Catia V5 ma do dyspozycji kilka opcji więcej niż...

**[4]** `Wyklad 2.pdf` — strona 30

> no główne „body”. Wyjątkiem są tu  konstrukcje spawane. 7. Niektóre dostępne operacje boolowskie nie są czułę na cechy tworzonych brył  (dodatnie/ujemne). Nie powinno się więc stosować funkcji „Add” i...

**[5]** `Wyklad 10.pdf` — strona 14

> Wymiarowanie w układzie 3D  Plik: BOM Table Listing.pdf PDF 3D – przykłady zastosowania – Spis części (Bill of Material) INŻYNIERSKIE PAKIETY CAD-CAM...

In [61]:
from IPython.display import display, Markdown

prompt_template ="""Based on the following context items, please answer the query.
Give yourself room to think by extracting relevant passages from the context before answering the query.
Don't return the thinking, only return the answer.
You must answer using only information explicitly present in the context.
If the answer is not directly stated or cannot be inferred with high confidence from the context, respond exactly:
"Nie znaleziono informacji w dostarczonych dokumentach."
Do not use prior knowledge.
Do not guess.
Do not supplement missing information.
Do not explain why information is missing.
Answer in Polish language only.
Use the following examples as reference for the ideal answer style.
Example 1:
Pytanie: Dlaczego Księżyc zawsze pokazuje tę samą stronę Ziemi?
Księżyc pokazuje Ziemi zawsze tę samą stronę, ponieważ jest związany pływowo z Ziemią. Oznacza to, że jego czas obrotu wokół własnej osi jest równy czasowi obiegu wokół Ziemi (około 27,3 dnia). W wyniku działania sił grawitacyjnych Ziemi rotacja Księżyca została w przeszłości spowolniona aż do osiągnięcia tego stanu równowagi.
Now use the following context items to answer this one user query only:
{context}
Relevant passages: <extract relevant passages from the context here>
Main User Query: {query}
Answer:\n"""

# Wybieramy losowo zapytanie z listy
random_query = np.random.choice(questions)

response, chunks = utils.answer_question(
        prompt_template=prompt_template,
        query=random_query,
        max_tokens=512,
        temp=0.1
)

display(Markdown(f"**Pytanie:** {random_query}"))
display(Markdown(f"**Odpowiedź:**\n\n{response}"))
display(Markdown("---\n**Źródła:**"))
for i, chunk in enumerate(chunks):
    excerpt = chunk['chunk'][:200].strip() + "..."
    display(Markdown(
        f"**[{i+1}]** `{chunk['filename']}` — strona {chunk['page_number'] + 1}\n\n"
        f"> {excerpt}"
    ))

**Pytanie:** Dlaczego wykrywanie małych egzoplanet jest trudniejsze niż dużych?

**Odpowiedź:**

Nie znaleziono informacji w dostarczonych dokumentach.

---
**Źródła:**

**[1]** `Wyklad 6.pdf` — strona 13

> raku ostrzy) i ciągłość drugiej pochodnej (ciągłość krzywizny), która  nie daje efektu wizualnego jak poprzednie ciągłości. Schemat składania krzywej B-sklejanej z krzywych Béziera i związki między pu...

**[2]** `Wyklad 2.pdf` — strona 30

> zygotowywany w szkicowniku powinien być możliwie najprostszy. Nie  oznacza to jednak, że powinien składać się jedynie z prostokątów i okręgów. Należy znaleźć  kompromis pomiędzy stopniem szczegółowośc...

**[3]** `Wyklad 2.pdf` — strona 18

> Szkicownik Szkicownik – sprawdzanie kompletności wymiarowania Każdy szkicownik powinien być w pełni zwymiarowany (wszytskie elementy geometryczne powinny  być wyświetlane w kolorze zielonym). Przy bar...

**[4]** `Wyklad 1.pdf` — strona 14

> y rozwiazan  parametrycznych (opartych o zdefiniowane zalżności wybranych relacji wymiarowych i  geometrycznych). Podobnie jak przy wczesniejszych programach możliwe jest korzystanie z  bogatych bibli...

**[5]** `Wyklad 1.pdf` — strona 15

> złożone niż w modelerach 3D. Należy tu wymienic narzedzia do zaawansowanego  modelowania powierzchniowego czy też analiz obliczeniowych, dużo szerszych niż część  wytrzymałosciowa np.: analizy przepł...